# Phase 6 — XLS-R Fine-Tuning on Speaker-Diverse Nepali (Colab T4)

Fine-tunes `gagan3012/wav2vec2-xlsr-nepali` (the audited checkpoint) on the
speaker-disjoint OpenSLR-54 train split, to close the generalization gap
measured in Phase 4 (4.91% WER in-domain vs 62.30% out-of-domain).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then
`Runtime -> Restart session`.

Tracking is **MLflow** (local `mlruns/` inside the repo -- synced to Drive if
you clone into Drive, which this notebook does, so runs survive disconnects).

## Step 1: mount Drive + get the code (branch `coursework-10phase`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
REPO = '/content/drive/MyDrive/NSTT-Lite'
if os.path.exists(REPO):
    !cd {REPO} && git fetch && git checkout coursework-10phase && git pull
else:
    !git clone -b coursework-10phase https://github.com/Rbimochan/NSTT-Lite.git {REPO}
%cd {REPO}

## Step 2: install deps (restart runtime after, then continue from Step 3)

In [ ]:
!pip install -q -r requirements.txt
print('If no red ResolutionImpossible error above: Runtime -> Restart session, then run from Step 3.')

## Step 3: GPU + data check
**SCREENSHOT this cell's output** (GPU + library versions -- rubric evidence).

In [ ]:
import os
os.chdir('/content/drive/MyDrive/NSTT-Lite')
import torch, transformers, datasets, mlflow
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- fix runtime type before continuing!')
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| datasets', datasets.__version__, '| mlflow', mlflow.__version__)
!wc -l data/manifests/train.jsonl data/manifests/val.jsonl data/manifests/test.jsonl

**Audio files check:** training reads `data/processed/*.wav` (built in Phase 3,
gitignored -- NOT in the repo). If the cell below shows 0 files, sync
`data/processed/` from your local machine to Drive first (~1.6GB), or rebuild it
on Colab by also syncing `data/openslr54_ne/` and running
`python -c "from pathlib import Path; from src.pipeline import run_slr54_pipeline; run_slr54_pipeline(Path('.'), target_hours=15.0, seed=42)"`.

In [ ]:
!ls data/processed/ 2>/dev/null | wc -l

## Step 4: full fine-tuning run (multi-hour)
Early stopping: patience 2 on val WER; 5-epoch ceiling; checkpoints each epoch to Drive.

In [ ]:
!python scripts/run_xlsr_train.py --output-dir models/xlsr-ft

### If Colab disconnects
Reconnect, re-run Steps 1-3, then resume (picks up the latest epoch checkpoint on Drive):

In [ ]:
#!python scripts/run_xlsr_train.py --output-dir models/xlsr-ft --resume true

## Step 5: MLflow evidence
**SCREENSHOT the MLflow UI** showing the phase6-finetune run's WER curve.

In [ ]:
# Colab can't open localhost directly; use the output-serving helper:
import subprocess
proc = subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', 'file:///content/drive/MyDrive/NSTT-Lite/mlruns', '--port', '5000'])
from google.colab import output
output.serve_kernel_port_as_window(5000)

## Done

Report back: final `eval_wer`, `global_step`, and the checkpoint path
(`models/xlsr-ft` on Drive). Phase 7 (before/after x in/out-of-domain
re-evaluation) runs against this checkpoint.